In [ ]:
📌 super() 的作用: 在子类中调用父类的方法。这里调用父类的 __init__ 来初始化 name 和 age，子类只需要处理自己特有的 student_id。

# Python 3 简写
super().__init__(name, age)

# Python 2 写法（兼容旧代码可能会看到）
super(Student, self).__init__(name, age)


class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

class Student(Person):
    def __init__(self, name, age, student_id):
        super().__init__(name, age)   # 第一空：super() 或 super(Student, self)
        self.student_id = student_id

s = Student("张三", 18, "S20240001")
print(s.name)       # 张三
print(s.student_id) # S20240001

In [ ]:
"""
classmethod 和 staticmethod 的区别:
"""
一、核心区别总览
Python 类中有三种方法类型：

类型	            装饰器	            第一个参数	                能访问什么
实例方法	        无	            self（self → 指向实例本身）	实例属性 + 类属性
类方法	        @classmethod	cls（class → 指向类本身）	    类属性 + 可创建新实例
静态方法	        @staticmethod	无	                        啥都访问不到（除非显式传参）
# @classmethod ___
class Dog:
    species = "犬类"          # 类属性（所有实例共享）

    def __init__(self, name):
        self.name = name       # 实例属性

    @classmethod              # ← 装饰器：把方法标记为"类方法"
    def get_species(cls):     # ← cls (class) 自动接收类本身
        return cls.species    # 可以访问/修改类属性


# 两种调用方式
print(Dog.get_species())      # 通过类调用 ✅ 推荐
d = Dog("旺财")
print(d.get_species())         # 通过实例调用 ✅ 也可以


# 典型应用场景：工厂方法
class Date:
    def __init__(self, year, month, day):
        self.year = year
        self.month = month
        self.day = day

    @classmethod
    def from_string(cls, date_str):
        """工厂方法：从字符串创建实例"""
        year, month, day = map(int, date_str.split("-"))
        return cls(year, month, day)   # 用 cls() 创建新实例！

    @classmethod
    def today(cls):
        """工厂方法：获取今天的日期"""
        import datetime
        today = datetime.date.today()
        return cls(today.year, today.month, today.day)

# 使用
d1 = Date(2026, 4, 28)           # 普通构造
d2 = Date.from_string("2026-04-28")  # 工厂方法
d3 = Date.today()                     # 工厂方法
# 关键优势：cls() 让继承也能正常工作——子类调用 from_string() 时返回的是子类的实例。


#@staticmethod —— 静态方法
class MathUtils:
    @staticmethod                    # ← 无 self，无 cls
    def add(a, b):
        return a + b

    @staticmethod
    def is_even(n):
        return n % 2 == 0
# 本质
#就是一个"恰好放在类里面"的普通函数。它和类的关系仅仅是命名空间归属，逻辑上完全不依赖类。

MathUtils.add(1, 2)      # 通过类调用 ✅
m = MathUtils()
m.add(1, 2)             # 通过实例也可以（但不推荐）


#典型应用场景
class Student:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @staticmethod
    def validate_score(score):   # 纯工具函数，不需要实例数据
        """验证成绩是否合法——纯逻辑，无状态"""
        if not isinstance(score, (int, float)):
            raise TypeError("成绩必须是数字")
        if not 0 <= score <= 100:
            raise ValueError("成绩必须在 0-100 之间")
        return True

# 使用：可以在不创建实例的情况下直接调用
Student.validate_score(88)   # True


"""
#
如何选择？


需要操作实例数据？  ──→  实例方法（普通方法）
       │
       否
       ▼
需要操作类属性 / 创建实例？  ──→  @classmethod
       │
       否
       ▼
只是相关工具函数，放在类里为了语义清晰？  ──→  @staticmethod

"""

In [ ]:
@staticmethod 根本不接收任何隐式参数。
实例方法 / 类方法 —— 有"隐式通道"

class Demo:
    count = 0

    @classmethod
    def get_count(cls):        # Python 自动把 Demo 传给 cls
        return cls.count       # cls.count → Demo.count → 拿到了！✅

# 调用 Demo.get_count() 时，Python 内部等价于：
# get_count(Demo)   ← 自动把类塞进来



class Demo:
    count = 0

    def show(self):            # Python 自动把实例传给 self
        return self.count      # self.count → 通过实例找到 count ✅

# 调用 d.show() 时，Python 内部等价于：
# show(d)   ← 自动把实例塞进来


# 它们都有一个隐式接收的参数（cls 或 self），这就是访问类/实例数据的"通道"

# 静态方法 —— 什么都没收到！
@staticmethod
def info():                    # 括号是空的！！没有 self，没有 cls！
    return "这是一个Demo类"
    # 这里写 return count  ❌ 报错：NameError: name 'count' is not defined


In [ ]:
# Python 属性查找链（MRO 的一部分）
# 当你写 a.count 时，Python 不是直接去 a 的字典里找，而是按顺序一层层往下找：

# code
"""a.count
   │
   ▼
┌─────────────────────────────┐
│ 第1步：找 实例自己 (a.__dict__) │
│                              │
│  a.__dict__ = {"name":"旺财"} │
│  有 count 吗？ → 没有 ❌        │
│         ↓                     │
│ 第2步：找 类本身 (Demo.__dict__)│
│                              │
│  Demo.__dict__ = {           │
│    "count": 3,               │  ← 找到了！✅ 停止！返回 3
│    "get_count": ...,          │
│    "info": ...                │
│  }                            │
└─────────────────────────────┘
用代码验证
"""


# python
class Demo:
    count = 3

    def __init__(self, name):
        self.name = name

a = Demo("旺财")

print(a.__dict__)      # {'name': '旺财'}     ← 实例自己的字典，没有 count
print(Demo.__dict__)   # {..., 'count': 3, ...} ← 类的字典里有 count

print(a.count)         # 3  ✅ 查找链找到了



#如果实例"覆盖"了类属性？
# python
class Demo:
    count = 3

a = Demo()
print(a.count)          # 3   （实例没有，找到类的）

a.count = 10            # 这行做了什么？
print(a.count)          # 10  ✅ 实例有了，直接返回
print(Demo.count)       # 3   ⚠️ 类的还是原来的！

# 此时：
# a.__dict__ = {"count": 10}    ← 实例现在有了自己的 count
# 查找时第1步就找到了，不会再去找类的了
"""
图解：

code
修改前:
  a.__dict__: {}                    → 没有 count
  Demo.__dict__: {"count": 3}       → 找到 3

a.count = 10 后:

  a.__dict__: {"count": 10}         → 第1步就找到了！返回 10 ✅ 不再往下找了
  Demo.__dict__: {"count": 3}       → 被遮挡了（但还在）

Demo.count                         # 直接查类，还是 3"""
# 一句话总结
#a.count 不是"直接拿"，而是一次查找之旅：先翻实例口袋 → 没有？再去翻类口袋 → 还没有？继续往上（父类）翻... 找到了就立刻返回。这就是 属性查找链（Attribute Lookup Chain）。







In [ ]:
"""
property 自带三个装饰器
当你用 @property 定义一个属性后，Python 会自动提供三个配套装饰器：
"""
class BankAccount:
    def __init__(self, balance):
        self._balance = balance

    @property                       # ① getter（读取）—— 内置
    def balance(self):
        return self._balance

    @balance.setter                 # ② setter（写入）—— 内置！
    def balance(self, value):       # 注意：方法名必须和上面一样！
        if value < 0:
            raise ValueError("余额不能为负!")
        self._balance = value

    @balance.deleter                # ③ deleter（删除）—— 内置！
    def balance(self):
        print("删除了 _balance!")
        del self._balance

"""
装饰器	        触发时机	                  作用
@property	读取时: acct.balance	        返回值（getter）
@xxx.setter	赋值时: acct.balance = 2000	设置值 + 验证（setter）
@xxx.deleter	del 时: del acct.balance	清理/删除（deleter）
"""



# 关键规则：方法名必须一致
@property
def balance(self):           # 名字叫 balance ...
    return self._balance

@balance.setter             # ... 所以这里必须是 balance.setter
def balance(self, value):   # 方法名也必须叫 balance！
    ...

# 如果写成：
# @balance.setter
# def money(self, value):   # ❌ 名字不一样，报错！



# 顺序固定
"""
@property 装饰了 balance()
       ↓
balance 现在是一个 property 对象了：
{
    fget: (getter函数),
    fset: None,            ← 还没有 setter
    fdel: None             ← 还没有 deleter
}
       ↓
@balance.setter 装饰了 balance()
       ↓
balance 这个 property 对象变成：
{
    fget: (getter函数),
    fset: (setter函数),    ← 有了！
    fdel: None
}
       ↓
@balance.deleter 装饰了 balance()
       ↓
最终完整版：
{
    fget: (getter函数),    ✅
    fset: (setter函数),    ✅
    fdel: (deleter函数)    ✅
}
"""

In [ ]:
"""
MRO（Method Resolution Order）—— 方法解析顺序
Python 用 C3 线性化算法决定查找顺序：

code
D 的 MRO 链：
D → B → C → A → object

d.show() 的查找过程：
① D 有 show()？  没有
② B 有 show()？  有！→ 输出 "B" ✅ 停止
（不会再往后找 C 和 A 了）
验证方式：

python
print(D.__mro__)
# (<class 'D'>, <class 'B'>, <class 'C'>, <class 'A'>, <class 'object'>)
简化记忆规则
从左到右，深度优先，每个类只出现一次，左边的优先。"""

In [ ]:
# try/except/finally 执行流程
def divide(a, b):
    try:
        result = a / b            # ① 正常执行
        return result             # ② return 值被"暂存"
    except ZeroDivisionError:
        print("异常!")            # 不执行（没抛异常）
    else:                          # ③ try 没异常才执行
        print("try块成功完成")
    finally:                       # ④ 无论怎样都最后执行
        print("清理工作")

divide(10, 2)

# 输出
# 清理工作      # finally 最后执行
# 5.0           # 之前暂存的 return 值最终返回


# 异常
divide(10, 0)
# 输出
# 被除数为0!    # except 捕获了异常
# 清理工作      # finally 依然最后执行
# None          # except 里 return None

# 完整流程图
"""
try 块
  │
  ├── 成功？
  │     │
  │     ├── 是 → else 块（可选）→ finally → 结束
  │     │
  │     └── 否 ↓
  │
  └── 匹配到对应的 except？
        │
        ├── 是 → except 处理 → finally → 结束
        │
        └── 否 → 异常向上抛出 → finally（如果存在）→ 程序崩溃
"""
# 最重要的规则
# finally 永远是最后执行的，即使前面有 return，也会先执行 finally，然后再返回。

def test():
    try:
        return "try的返回值"      # ← 不会立即返回！
    finally:
        print("finally先执行！")   # ← 先干这个！

result = test()
# 输出：
#   finally先执行！
# 然后 result = "try的返回值"
